# Exercise 1. Steering LLMs Away from Harmful Content
A major concern of generative AI is its potential to produce content misaligned with human values, from reinforcing harmful stereotypes to spreading conspiracy theories.

```{figure} ../figures/class8/chatgpt-evil-versus-good.png
---
name: evil versus good llm
width: 100%
---
AI-generated, modified by me :) 
```

How do we deal with this? We could try *prompt-engineering* as we did in [Class 6](../../book/class6/001_prompting.ipynb) or [Reinforcement Learning with Human Feedback (RLHF)](https://magazine.sebastianraschka.com/i/161572341/rlhf-basics-where-it-all-started). Yet, prompting may prove to be ineffective or instable, and RLHF is a costly approach, both in terms of time and compute. 

## 1.1 Intro to Steering Vectors
An intriguing training-free alternative is to manipulate the transformer’s **activation space**, the internal representations computed at each layer. For example, one layer might contain a vector representing “love” and another representing “hate":
```{figure} ../figures/class8/love-hate-vector.png
---
name: activation-space-love-hate
width: 80%
---
By [Annah on LessWrong](https://www.lesswrong.com/posts/ndyngghzFY388Dnew/implementing-activation-steering)
```

The idea is that if we know that these internal vectors exist, we can also *use* them to impact model behaviour. In practice, we compute a *steering vector* that can allow us to push the model toward one direction or the other:
```{figure} ../figures/class8/steering-vector-compute.png
---
name: steering-vector-compute
width: 100%
---
Re-interpretation. Originally by [Anastasia Borovykh](https://youtu.be/cp-YSyc5aW8?si=tkgji879u6kChajs&t=116).
```
To do this, we use pairs of prompts (as shown in the figure above), where one prompt includes the target property (A) we wish to steer toward or away from, and the other (B) either represents the opposite (a contrastive prompt) or simply lacks that property.

:::{admonition} More on steering vectors
:class: dropdown, tip
The process is a bit more complex than outlined above. For example, you need to decide which layer to compute the steering vector from, and you may choose to use a normalized vector rather than the raw one. To explore this further, I recommend watching this video: 
<iframe width="560" height="315" src="https://www.youtube.com/embed/cp-YSyc5aW8?si=JpOToi4AJAMYbhXP" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
:::

## 1.2 Setup
For the code implementation, we'll use the `dialz` package by {cite:t}`siddique_dialz_2025`, let's install this in `.venv`:
```bash
source .venv/bin/activate
pip install dialz
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers 
```

Finally, let's import what we need: 